# 001 — Fine-Tuning with Unsloth & QLoRA

Fine-tune a pre-trained LLM using QLoRA (4-bit quantized LoRA) with the
Unsloth library for accelerated training. Uses Gemma 3 4B IT on the Alpaca
instruction dataset by default.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook — no external library imports
from a shared `src/` package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Model
model_name: str = "unsloth/gemma-3-4b-it"  # Hugging Face model ID
dataset_name: str = "yahma/alpaca-cleaned"  # Hugging Face dataset ID
max_seq_length: int = 2048
load_in_4bit: bool = True

# LoRA configuration
lora_r: int = 16           # LoRA rank
lora_alpha: int = 32       # LoRA alpha scaling factor
lora_dropout: float = 0.05
target_modules: list[str] = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training
max_steps: int = 60                        # quick demo; increase for real runs
per_device_train_batch_size: int = 4
gradient_accumulation_steps: int = 4
learning_rate: float = 2e-4
warmup_steps: int = 10
logging_steps: int = 5
lr_scheduler_type: str = "cosine"

# Dataset
dataset_sample_size: int = 1000
test_size: float = 0.1
seed: int = 42

# Inference test prompts
test_prompts: list[str] = [
    "Explain the difference between a list and a tuple in Python.",
    "Write a haiku about machine learning.",
    "What are three benefits of exercise?",
]

# Model saving
save_merged_model: bool = True

# Output paths
metrics_json_path: str = "outputs/metrics/metrics.json"
model_output_path: str = "outputs/models/lora_adapter"
merged_output_path: str = "outputs/models/merged_model"
plots_dir: str = "outputs/plots"
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports & setup
# ---------------------------------------------------------------------------
import json
import math
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

# Ensure output dirs exist
for d in ["outputs/runs", "outputs/plots", "outputs/models", "outputs/metrics"]:
    Path(d).mkdir(parents=True, exist_ok=True)

run_start = datetime.now(timezone.utc)
print(f"Run started at {run_start.isoformat()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1 — Load Base Model

Load the pre-trained model with 4-bit quantization (QLoRA). This reduces
memory usage by ~4x compared to full precision, making it possible to
fine-tune large models on consumer GPUs.

In [ ]:
# ---------------------------------------------------------------------------
# Load pre-trained model + tokenizer
# ---------------------------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Max sequence length: {max_seq_length}")
print(f"Quantization: {'4-bit' if load_in_4bit else 'none'}")

## 2 — Configure LoRA Adapters

LoRA (Low-Rank Adaptation) adds small trainable matrices to the model's
attention and feed-forward layers. Only these adapters are trained, keeping
the base model frozen.

Key parameters:
- **rank (r):** Dimension of the low-rank matrices. Higher = more capacity but more memory.
- **alpha:** Scaling factor. The effective learning rate scales as `alpha / r`.
- **target_modules:** Which layers to attach adapters to.

In [ ]:
# ---------------------------------------------------------------------------
# Attach LoRA adapters
# ---------------------------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules,
    use_gradient_checkpointing="unsloth",
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
pct = 100.0 * trainable_params / total_params

print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({pct:.2f}%)")
print(f"LoRA rank: {lora_r}, alpha: {lora_alpha}, dropout: {lora_dropout}")
print(f"Target modules: {target_modules}")

## 3 — Load and Prepare Dataset

Load the Alpaca instruction-following dataset, subsample for quick
experimentation, and format each example with the Alpaca prompt template.

In [ ]:
# ---------------------------------------------------------------------------
# Load and subsample dataset
# ---------------------------------------------------------------------------
raw_dataset = load_dataset(dataset_name, split="train")
print(f"Full dataset size: {len(raw_dataset):,}")

if dataset_sample_size and dataset_sample_size < len(raw_dataset):
    raw_dataset = raw_dataset.shuffle(seed=seed).select(range(dataset_sample_size))
    print(f"Subsampled to: {len(raw_dataset):,}")

print(f"\nColumns: {raw_dataset.column_names}")
print(f"\nExample:")
for k, v in raw_dataset[0].items():
    print(f"  {k}: {v[:200] if isinstance(v, str) else v}")

In [ ]:
# ---------------------------------------------------------------------------
# Format with Alpaca template + train/eval split
# ---------------------------------------------------------------------------
ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

eos_token = tokenizer.eos_token


def format_alpaca(example):
    text = ALPACA_TEMPLATE.format(
        instruction=example["instruction"],
        input=example.get("input", ""),
        output=example["output"],
    )
    return {"text": text + eos_token}


formatted_dataset = raw_dataset.map(format_alpaca)

# Train / eval split
split = formatted_dataset.train_test_split(test_size=test_size, seed=seed)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Eval samples:  {len(eval_dataset):,}")
print(f"\nFormatted example (first 500 chars):")
print(train_dataset[0]["text"][:500])

## 4 — Token Length Analysis

Visualize the distribution of tokenized sequence lengths to verify
that `max_seq_length` is appropriate for this dataset.

In [ ]:
# ---------------------------------------------------------------------------
# Token length distribution
# ---------------------------------------------------------------------------
sample_texts = train_dataset["text"][:500]
token_lengths = [len(tokenizer.encode(t)) for t in sample_texts]

print(f"Token length stats (sample of {len(sample_texts)}):")
print(f"  Min:    {min(token_lengths)}")
print(f"  Max:    {max(token_lengths)}")
print(f"  Mean:   {np.mean(token_lengths):.0f}")
print(f"  Median: {np.median(token_lengths):.0f}")
print(f"  95th %%: {np.percentile(token_lengths, 95):.0f}")
print(f"  > max_seq_length ({max_seq_length}): {sum(1 for l in token_lengths if l > max_seq_length)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(token_lengths, bins=50, color="#2196F3", edgecolor="white", alpha=0.8)
axes[0].axvline(max_seq_length, color="red", linestyle="--", linewidth=2,
               label=f"max_seq_length={max_seq_length}")
axes[0].set_xlabel("Token Length")
axes[0].set_ylabel("Count")
axes[0].set_title("Token Length Distribution")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_lengths = np.sort(token_lengths)
cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
axes[1].plot(sorted_lengths, cumulative, color="#9C27B0", linewidth=2)
axes[1].axvline(max_seq_length, color="red", linestyle="--", linewidth=2,
               label=f"max_seq_length={max_seq_length}")
axes[1].set_xlabel("Token Length")
axes[1].set_ylabel("Cumulative Fraction")
axes[1].set_title("Cumulative Token Length Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(f"{plots_dir}/token_length_distribution.png", dpi=120)
plt.show()
print(f"Saved \u2192 {plots_dir}/token_length_distribution.png")

## 5 — Training

Train the LoRA adapters using the SFTTrainer from TRL. The base model
weights remain frozen — only the low-rank adapter matrices are updated.

In [ ]:
# ---------------------------------------------------------------------------
# Configure training
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="outputs/training_checkpoints",
    max_steps=max_steps,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    warmup_steps=warmup_steps,
    logging_steps=logging_steps,
    lr_scheduler_type=lr_scheduler_type,
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=seed,
    eval_strategy="steps",
    eval_steps=logging_steps,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
)

print("Training configuration:")
print(f"  Max steps: {max_steps}")
print(f"  Batch size: {per_device_train_batch_size}")
print(f"  Gradient accumulation: {gradient_accumulation_steps}")
print(f"  Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps}")
print(f"  Learning rate: {learning_rate}")
print(f"  Scheduler: {lr_scheduler_type}")
print(f"  Precision: {'bf16' if training_args.bf16 else 'fp16'}")

In [ ]:
# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
gpu_mem_before = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

train_start = time.time()
train_result = trainer.train()
train_elapsed = time.time() - train_start

gpu_mem_after = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"\nTraining complete in {train_elapsed:.1f}s")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  GPU memory before: {gpu_mem_before:.2f} GB")
print(f"  GPU peak memory:   {gpu_mem_after:.2f} GB")

## 6 — Training Loss Visualization

In [ ]:
# ---------------------------------------------------------------------------
# Plot training loss + learning rate schedule
# ---------------------------------------------------------------------------
log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]
lr_steps = [e["step"] for e in log_history if "learning_rate" in e]
lr_values = [e["learning_rate"] for e in log_history if "learning_rate" in e]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Loss
ax1.plot(train_steps, train_losses, color="#2196F3", alpha=0.4, linewidth=1, label="Train loss")
if len(train_losses) >= 3:
    window = min(5, len(train_losses))
    rolling = np.convolve(train_losses, np.ones(window) / window, mode="valid")
    rolling_steps = train_steps[window - 1:]
    ax1.plot(rolling_steps, rolling, color="#4CAF50", linewidth=2, label=f"Rolling mean (w={window})")
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper left")

# Learning rate on secondary axis
if lr_values:
    ax2 = ax1.twinx()
    ax2.plot(lr_steps, lr_values, color="#FF9800", linewidth=1.5, linestyle="--", label="Learning rate")
    ax2.set_ylabel("Learning Rate")
    ax2.legend(loc="upper right")

ax1.set_title("Training Loss & Learning Rate")
fig.tight_layout()
fig.savefig(f"{plots_dir}/training_loss.png", dpi=120)
plt.show()
print(f"Saved \u2192 {plots_dir}/training_loss.png")

## 7 — Evaluation

Evaluate the fine-tuned model on the held-out set and compute perplexity.

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate on held-out set
# ---------------------------------------------------------------------------
eval_result = trainer.evaluate()
eval_loss = eval_result["eval_loss"]
perplexity = math.exp(eval_loss)

# Get final train loss from log history
final_train_loss = train_result.training_loss

print(f"Eval loss:    {eval_loss:.4f}")
print(f"Perplexity:   {perplexity:.2f}")
print(f"Train loss:   {final_train_loss:.4f}")

# Bar chart: train vs eval loss
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Train Loss", "Eval Loss"],
    [final_train_loss, eval_loss],
    color=["#2196F3", "#FF9800"],
    edgecolor="white",
)
for bar, val in zip(bars, [final_train_loss, eval_loss]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", fontsize=11)
ax.set_ylabel("Loss")
ax.set_title("Train vs Eval Loss")
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(f"{plots_dir}/train_vs_eval_loss.png", dpi=120)
plt.show()
print(f"Saved \u2192 {plots_dir}/train_vs_eval_loss.png")

## 8 — Sample Generation

Generate responses from the fine-tuned model to qualitatively assess
instruction-following ability.

In [ ]:
# ---------------------------------------------------------------------------
# Generate sample responses
# ---------------------------------------------------------------------------
FastLanguageModel.for_inference(model)

generated_samples = []

for i, prompt_text in enumerate(test_prompts):
    formatted_prompt = ALPACA_TEMPLATE.format(
        instruction=prompt_text,
        input="",
        output="",
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated_samples.append({"prompt": prompt_text, "response": response.strip()})

    print(f"\n{'=' * 60}")
    print(f"Prompt {i + 1}: {prompt_text}")
    print(f"{'─' * 60}")
    print(response.strip())

# Response length chart
response_lengths = [len(s["response"]) for s in generated_samples]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(response_lengths)), response_lengths, color="#4CAF50", edgecolor="white")
ax.set_xlabel("Prompt Index")
ax.set_ylabel("Response Length (chars)")
ax.set_title("Response Lengths")
ax.set_xticks(range(len(response_lengths)))
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(f"{plots_dir}/response_lengths.png", dpi=120)
plt.show()
print(f"\nSaved \u2192 {plots_dir}/response_lengths.png")

## 9 — Save Model

Save the LoRA adapters (small, portable) and optionally merge them
into a full 16-bit model for direct inference without PEFT.

In [ ]:
# ---------------------------------------------------------------------------
# Save LoRA adapters
# ---------------------------------------------------------------------------
Path(model_output_path).mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_output_path)
tokenizer.save_pretrained(model_output_path)

adapter_size = sum(
    f.stat().st_size for f in Path(model_output_path).rglob("*") if f.is_file()
) / 1e6
print(f"LoRA adapters saved \u2192 {model_output_path} ({adapter_size:.1f} MB)")

In [ ]:
# ---------------------------------------------------------------------------
# Optionally save merged 16-bit model
# ---------------------------------------------------------------------------
merged_size_mb = None
if save_merged_model:
    Path(merged_output_path).mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(
        merged_output_path,
        tokenizer,
        save_method="merged_16bit",
    )
    merged_size_mb = sum(
        f.stat().st_size for f in Path(merged_output_path).rglob("*") if f.is_file()
    ) / 1e6
    print(f"Merged model saved \u2192 {merged_output_path} ({merged_size_mb:.1f} MB)")
else:
    print("Skipping merged model save (save_merged_model=False)")

# Next step: for GGUF export, use:
# model.save_pretrained_gguf("outputs/models/gguf", tokenizer, quantization_method="q4_k_m")

## 9.1 — LoRA Anatomy: What Gets Trained and Why

LoRA (Low-Rank Adaptation) is mathematically elegant: instead of updating a 
full weight matrix W ∈ ℝ^(d×k), it freezes W and learns two small matrices:

```
W_new = W + α/r · B·A    where  A ∈ ℝ^(r×k),  B ∈ ℝ^(d×r)
```

The update BA has rank ≤ r, making it a **low-rank approximation** of the 
true weight update. With r=16, we update `d×16 + 16×k` parameters instead 
of `d×k` — a 64× reduction for a 512×512 matrix.

**Why it works:**
- Pre-trained weights already encode rich world knowledge  
- Fine-tuning only needs to *redirect* this knowledge, not relearn it
- The true fine-tuning gradient has low intrinsic rank for most NLP tasks

**The rank hyperparameter `r`:**
- Small r (4-8): very parameter-efficient, fast, may underfit complex tasks
- Medium r (16-32): standard choice, balances efficiency and expressivity  
- Large r (64-128): approaches full fine-tuning, slower, more memory

**Alpha scaling factor:**
The effective learning rate for LoRA layers = lr × α/r.  
Setting α = 2r (common convention) gives an effective multiplier of 2×.

In [ ]:
# ---------------------------------------------------------------------------
# LoRA Parameter Analysis and Efficiency Visualization
# ---------------------------------------------------------------------------
import math

# Analyze which layers got LoRA adapters
print("LoRA Configuration Analysis")
print("=" * 55)
print(f"  Model           : {model_name}")
print(f"  Rank (r)        : {lora_r}")
print(f"  Alpha (α)       : {lora_alpha}")
print(f"  Effective scale : α/r = {lora_alpha/lora_r:.2f}x")
print(f"  Target modules  : {target_modules}")
print(f"  Dropout         : {lora_dropout}")
print()

# Count parameters per LoRA layer
print("Per-layer LoRA parameter count:")
lora_param_counts = {}
total_lora_params = 0
for name, param in model.named_parameters():
    if param.requires_grad and 'lora' in name.lower():
        count = param.numel()
        layer_type = name.split('.')[-2] if '.' in name else name
        if layer_type not in lora_param_counts:
            lora_param_counts[layer_type] = 0
        lora_param_counts[layer_type] += count
        total_lora_params += count

for layer_name, count in sorted(lora_param_counts.items(), key=lambda x: -x[1]):
    print(f"  {layer_name:<20}: {count:>12,} params")
print(f"  {'Total LoRA':<20}: {total_lora_params:>12,} params")
print()

# Rank sensitivity table
print("LoRA Rank Sensitivity (theoretical parameter counts):")
print(f"{'Rank (r)':>10} | {'Params':>15} | {'vs Full FT':>12} | {'Reduction':>10}")
print("-" * 55)
for r_val in [4, 8, 16, 32, 64, 128]:
    # Approximate: 2 matrices per module, each r * hidden_dim
    # For all target modules combined
    est_params = total_lora_params * r_val / lora_r  # scale from current r
    reduction = total_params / max(est_params, 1)
    pct_trainable = 100 * est_params / total_params
    print(f"{r_val:>10} | {int(est_params):>15,} | {pct_trainable:>11.3f}% | {reduction:>9.1f}x")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Parameter breakdown pie
ax = axes[0]
frozen_params = total_params - trainable_params
sizes = [trainable_params, frozen_params]
labels = [f'Trainable\n({trainable_params:,})\n{pct:.2f}%',
          f'Frozen\n({frozen_params:,})\n{100-pct:.2f}%']
colors_pie = ['#FF9800', '#E3F2FD']
wedges, texts = ax.pie(sizes, labels=labels, colors=colors_pie,
                        startangle=90, textprops={'fontsize': 9})
wedges[0].set_edgecolor('#FF5722')
wedges[0].set_linewidth(2)
ax.set_title(f'Parameter Efficiency\n{model_name.split("/")[-1]}', fontsize=11)

# Plot 2: Rank vs trainable parameters
ax = axes[1]
r_vals = [4, 8, 16, 32, 64, 128]
param_counts = [total_lora_params * r / lora_r for r in r_vals]
pct_trainable_vals = [100 * p / total_params for p in param_counts]
ax.plot(r_vals, pct_trainable_vals, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(lora_r, color='tomato', linestyle='--', linewidth=2,
           label=f'Current r={lora_r} ({pct:.2f}%)')
ax.set_xlabel('LoRA Rank (r)', fontsize=11)
ax.set_ylabel('Trainable Parameters (%)', fontsize=11)
ax.set_title('Rank vs Parameter Count\n(log scale)', fontsize=11)
ax.set_xscale('log', base=2)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Annotate current rank
current_pct = 100 * total_lora_params / total_params
ax.annotate(f'r={lora_r}\n{current_pct:.2f}%', xy=(lora_r, current_pct),
            xytext=(lora_r * 1.5, current_pct * 1.3),
            fontsize=9, color='tomato',
            arrowprops=dict(arrowstyle='->', color='tomato'))

# Plot 3: Memory comparison - QLoRA vs full finetuning estimate
ax = axes[2]
# Estimate: full FT requires ~4 bytes per param (fp32) + optimizer states (2x for AdamW)
# QLoRA: base model 4-bit quantized (0.5 bytes per param) + LoRA params (fp16 = 2 bytes)
bytes_per_param_full = 4 * 3  # fp32 params + Adam m + v (3 copies)
bytes_per_param_qlora_base = 0.5  # 4-bit quantization
bytes_per_param_qlora_lora = 2 * 3  # fp16 LoRA + Adam m + v

full_ft_gb = total_params * bytes_per_param_full / 1e9
qlora_base_gb = total_params * bytes_per_param_qlora_base / 1e9
qlora_lora_gb = trainable_params * bytes_per_param_qlora_lora / 1e9
qlora_total_gb = qlora_base_gb + qlora_lora_gb

methods = ['Full Fine-tuning\n(FP32 + AdamW)', 'QLoRA\n(4-bit base + LoRA)']
memory_gb = [full_ft_gb, qlora_total_gb]
bar_colors = ['#EF5350', '#66BB6A']
bars = ax.bar(methods, memory_gb, color=bar_colors, edgecolor='white', width=0.5)
for bar, mem in zip(bars, memory_gb):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{mem:.1f} GB', ha='center', va='bottom', fontsize=11, fontweight='bold')

reduction_factor = full_ft_gb / qlora_total_gb
ax.set_ylabel('Estimated GPU Memory (GB)', fontsize=11)
ax.set_title(f'Memory Efficiency\nQLoRA is ~{reduction_factor:.1f}x more memory-efficient', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, full_ft_gb * 1.2)

# Add breakdown annotation for QLoRA
ax.annotate(f'Base model: {qlora_base_gb:.1f} GB\nLoRA adapters: {qlora_lora_gb:.1f} GB',
            xy=(1, qlora_total_gb / 2), xytext=(0.5, qlora_total_gb * 0.6),
            fontsize=9, ha='center',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
_lora_path = os.path.join(plots_dir, 'lora_analysis.png')
fig.savefig(_lora_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_lora_path}')
print(f'\nKey efficiency metric: QLoRA trains {pct:.2f}% of parameters')
print(f'while achieving comparable fine-tuning quality for most instruction-following tasks.')

## 9.2 — Learning Rate Schedule Analysis

The **learning rate schedule** is one of the most critical training decisions 
for LLM fine-tuning. Poor scheduling leads to:

- **Too high LR:** loss spikes, training instability, catastrophic forgetting
- **Too low LR:** slow convergence, wasted compute
- **No warmup:** initial gradient explosions (the first tokens see randomly 
  initialized LoRA weights — warmup prevents them from taking a huge step)

The **cosine schedule with warmup** used here:

```
Step 0 → warmup_steps:   LR linearly ramps from 0 → peak_lr
Step warmup → max_steps: LR follows cosine decay from peak_lr → 0
```

This is the standard for LLM fine-tuning because:
1. **Warmup** prevents early instability
2. **Cosine decay** provides graceful annealing — the model converges 
   smoothly rather than stopping abruptly
3. The minimum LR is ~0 at `max_steps`, so the final model is at a stable local minimum

**Key relationship:** `effective_batch_size = batch_size × grad_accumulation_steps`  
Larger effective batch → smoother gradients → can use higher peak LR.

In [ ]:
# ---------------------------------------------------------------------------
# Learning Rate Schedule Visualization + Training Loss Analysis
# ---------------------------------------------------------------------------
import math

# Reconstruct the LR schedule
def cosine_schedule_with_warmup(step, warmup_steps, max_steps, peak_lr, min_lr=0.0):
    if step < warmup_steps:
        return peak_lr * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(max_steps - warmup_steps, 1)
    cosine_val = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + (peak_lr - min_lr) * cosine_val

steps = np.arange(0, max_steps + 1)
lr_values = [cosine_schedule_with_warmup(s, warmup_steps, max_steps, learning_rate)
             for s in steps]

# Extract training loss from trainer log history
log_history = trainer.state.log_history if hasattr(trainer, 'state') else []
train_steps = [entry['step'] for entry in log_history if 'loss' in entry]
train_losses = [entry['loss'] for entry in log_history if 'loss' in entry]
train_lrs = [entry.get('learning_rate', None) for entry in log_history if 'loss' in entry]

effective_batch = per_device_train_batch_size * gradient_accumulation_steps
print(f"Training Configuration:")
print(f"  Model                 : {model_name}")
print(f"  Peak learning rate    : {learning_rate:.2e}")
print(f"  Warmup steps          : {warmup_steps} ({100*warmup_steps/max_steps:.1f}% of training)")
print(f"  LR scheduler          : {lr_scheduler_type}")
print(f"  Effective batch size  : {effective_batch}")
print(f"  Total steps           : {max_steps}")
print(f"  Training time         : {train_elapsed:.1f}s ({train_elapsed/60:.1f} min)")
print(f"  Steps/second          : {max_steps/max(train_elapsed,1):.2f}")
print()
print(f"Loss summary:")
print(f"  Initial loss (first logged): {train_losses[0]:.4f}" if train_losses else "  No training logs found")
print(f"  Final loss              : {train_losses[-1]:.4f}" if train_losses else "")
if len(train_losses) > 1:
    total_drop = train_losses[0] - train_losses[-1]
    pct_drop = 100 * total_drop / max(train_losses[0], 1e-6)
    print(f"  Absolute drop           : {total_drop:.4f} ({pct_drop:.1f}%)")
print(f"  Eval loss               : {eval_loss:.4f}")
print(f"  Perplexity              : {perplexity:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: LR schedule
ax = axes[0]
ax.plot(steps, [lr * 1e4 for lr in lr_values], color='steelblue', linewidth=2)
ax.axvspan(0, warmup_steps, alpha=0.15, color='orange', label=f'Warmup ({warmup_steps} steps)')
ax.set_xlabel('Training Step', fontsize=11)
ax.set_ylabel(f'Learning Rate (×10⁻⁴)', fontsize=11)
ax.set_title(f'Cosine LR Schedule with Warmup\nPeak={learning_rate:.2e}, Warmup={warmup_steps} steps',
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Training loss over steps
ax = axes[1]
if train_steps and train_losses:
    ax.plot(train_steps, train_losses, 'o-', color='tomato', linewidth=2,
            markersize=6, label='Train loss (per logging step)')
    # Smoothed trend (if enough points)
    if len(train_losses) > 5:
        from scipy.ndimage import uniform_filter1d
        try:
            smooth = uniform_filter1d(train_losses, size=3)
            ax.plot(train_steps, smooth, color='darkred', linewidth=1.5,
                    linestyle='--', alpha=0.7, label='Smoothed')
        except ImportError:
            pass
    ax.axhline(eval_loss, color='steelblue', linestyle='--', linewidth=1.5,
               label=f'Eval loss={eval_loss:.4f} (perplexity={perplexity:.2f})')
    ax.set_xlabel('Training Step', fontsize=11)
    ax.set_ylabel('Cross-Entropy Loss', fontsize=11)
    ax.set_title(f'Training Loss Curve\nFinal: {train_losses[-1]:.4f} → Perplexity={perplexity:.2f}',
                 fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    print(f"\nThe gap between train_loss={train_losses[-1]:.4f} and eval_loss={eval_loss:.4f} shows generalization.")
else:
    ax.text(0.5, 0.5, 'Training loss not logged\n(check logging_steps parameter)',
            ha='center', va='center', transform=ax.transAxes, fontsize=11)

plt.tight_layout()
_lr_path = os.path.join(plots_dir, 'lr_schedule_and_loss.png')
fig.savefig(_lr_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_lr_path}')

## 9.3 — Generation Quality: Response Analysis

Quantitative metrics like **perplexity** measure how well the model assigns 
probability to the training distribution, but they don't capture *quality* 
of generation. A model could have low perplexity while producing repetitive 
or incoherent responses.

**Analyzing generated samples reveals:**
1. **Instruction following:** does the model complete the requested task?
2. **Verbosity:** are responses appropriately detailed or too short/long?
3. **Lexical diversity:** is the model using varied vocabulary or repeating itself?
4. **Format adherence:** does the model follow the Alpaca template structure?

These qualitative signals complement the quantitative perplexity metric 
and are essential for real-world deployment decisions.

In [ ]:
# ---------------------------------------------------------------------------
# Generation Quality Analysis — Lexical Diversity and Response Statistics
# ---------------------------------------------------------------------------
import re
import pandas as pd

def _tokenize_simple(text):
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r'\b\w+\b', text.lower())

if generated_samples:
    analysis = []
    for sample in generated_samples:
        prompt = sample['prompt']
        response = sample['response']
        tokens = _tokenize_simple(response)
        unique_tokens = set(tokens)
        n_words = len(tokens)
        n_unique = len(unique_tokens)
        ttr = n_unique / max(n_words, 1)  # type-token ratio (lexical diversity)

        # Check if response seems complete (ends with sentence terminator)
        is_complete = response.strip()[-1] in '.!?)' if response.strip() else False

        # Count sentences
        sentences = re.split(r'[.!?]+', response)
        n_sentences = len([s for s in sentences if len(s.strip()) > 10])

        analysis.append({
            'prompt': prompt[:60] + '...' if len(prompt) > 60 else prompt,
            'response_chars': len(response),
            'response_words': n_words,
            'unique_words': n_unique,
            'type_token_ratio': round(ttr, 3),
            'n_sentences': n_sentences,
            'is_complete': is_complete,
        })

    print("Generated Sample Analysis:")
    print("=" * 70)
    for i, (sample, a) in enumerate(zip(generated_samples, analysis)):
        print(f"\n[Sample {i+1}] Prompt: {a['prompt']}")
        print(f"  Response length : {a['response_chars']} chars, {a['response_words']} words")
        print(f"  Lexical diversity (TTR): {a['type_token_ratio']:.3f} "
              f"(0=repetitive, 1=all unique; typical: 0.4–0.7)")
        print(f"  Sentences: {a['n_sentences']}, Complete: {a['is_complete']}")
        print(f"  Response preview: {sample['response'][:200]}...")

    # Perplexity interpretation
    print(f"\n{'='*55}")
    print(f"Perplexity Analysis:")
    print(f"  Perplexity       : {perplexity:.2f}")
    print(f"  Interpretation   :")
    if perplexity < 5:
        interp = "Excellent — model is very confident in its predictions"
    elif perplexity < 15:
        interp = "Good — typical for well-trained instruction-following models"
    elif perplexity < 50:
        interp = "Moderate — model has learned something but may need more training"
    else:
        interp = "High — model may be underfitting; consider more steps or higher rank"
    print(f"    {interp}")
    print(f"  Note: Perplexity = e^(eval_loss) = e^{eval_loss:.4f} = {perplexity:.2f}")

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Response statistics bar chart
    ax = axes[0]
    if analysis:
        categories = [f'Sample {i+1}' for i in range(len(analysis))]
        word_counts = [a['response_words'] for a in analysis]
        ttr_vals    = [a['type_token_ratio'] for a in analysis]

        x = np.arange(len(categories))
        width = 0.35
        bars = ax.bar(x - width/2, word_counts, width, label='Response words',
                      color='steelblue', alpha=0.8)
        ax2_gen = ax.twinx()
        ax2_gen.bar(x + width/2, ttr_vals, width, label='Lexical diversity (TTR)',
                    color='darkorange', alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(categories, fontsize=10)
        ax.set_ylabel('Response word count', fontsize=10)
        ax2_gen.set_ylabel('Type-Token Ratio (diversity)', fontsize=10, color='darkorange')
        ax2_gen.set_ylim(0, 1.2)
        ax.set_title('Response Statistics per Sample\n'
                     'TTR ≈ 0.5 is typical for natural language', fontsize=11)
        ax.legend(loc='upper left', fontsize=9)
        ax2_gen.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')

    # Plot 2: Training efficiency summary
    ax = axes[1]
    metrics_viz = {
        'Steps': max_steps,
        'Train time (s)': int(train_elapsed),
        'Perplexity': round(perplexity, 1),
        'Trainable\nparams (M)': round(trainable_params / 1e6, 1),
        'GPU memory\n(GB)': round(gpu_mem_after, 1) if gpu_mem_after > 0 else 0,
    }
    x_pos = np.arange(len(metrics_viz))
    colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
    bars_eff = ax.bar(x_pos, list(metrics_viz.values()), color=colors_bar, alpha=0.8, edgecolor='white')
    for bar, val in zip(bars_eff, metrics_viz.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(list(metrics_viz.values())) * 0.02,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(list(metrics_viz.keys()), fontsize=9)
    ax.set_title('Training Run Summary\n(all key metrics in one view)', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(list(metrics_viz.values())) * 1.2)

    plt.tight_layout()
    _gen_path = os.path.join(plots_dir, 'generation_quality.png')
    fig.savefig(_gen_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {_gen_path}')
else:
    print("No generated samples available for analysis.")
    print("Set test_prompts parameter to generate sample outputs.")

## 10 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------
gpu_info = {}
if torch.cuda.is_available():
    gpu_info = {
        "device_name": torch.cuda.get_device_name(0),
        "total_vram_gb": round(torch.cuda.get_device_properties(0).total_mem / 1e9, 2),
        "peak_memory_gb": round(gpu_mem_after, 2),
    }

metrics_output = {
    "run_metadata": {
        "timestamp": run_start.isoformat(),
        "model_name": model_name,
        "dataset_name": dataset_name,
        "dataset_sample_size": dataset_sample_size,
        "max_seq_length": max_seq_length,
        "load_in_4bit": load_in_4bit,
        "seed": seed,
        "n_train": len(train_dataset),
        "n_eval": len(eval_dataset),
    },
    "lora_config": {
        "r": lora_r,
        "alpha": lora_alpha,
        "dropout": lora_dropout,
        "target_modules": target_modules,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_pct": round(pct, 2),
    },
    "training": {
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": learning_rate,
        "lr_scheduler_type": lr_scheduler_type,
        "warmup_steps": warmup_steps,
        "train_loss": round(final_train_loss, 4),
        "train_runtime_s": round(train_elapsed, 1),
    },
    "evaluation": {
        "eval_loss": round(eval_loss, 4),
        "perplexity": round(perplexity, 2),
    },
    "model_outputs": {
        "lora_adapter_path": model_output_path,
        "lora_adapter_size_mb": round(adapter_size, 1),
        "merged_model_path": merged_output_path if save_merged_model else None,
        "merged_model_size_mb": round(merged_size_mb, 1) if merged_size_mb else None,
    },
    "generated_samples": generated_samples,
    "gpu_info": gpu_info,
}

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, "w") as f:
    json.dump(metrics_output, f, indent=2, default=str)

print(f"Metrics JSON saved \u2192 {metrics_json_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("RUN COMPLETE")
print("=" * 60)
print(f"  Model:              {model_name}")
print(f"  Dataset:            {dataset_name} ({len(train_dataset) + len(eval_dataset)} samples)")
print(f"  LoRA rank:          {lora_r}")
print(f"  Trainable params:   {trainable_params:,} ({pct:.2f}%)")
print(f"  Training steps:     {max_steps}")
print(f"  Train loss:         {final_train_loss:.4f}")
print(f"  Eval loss:          {eval_loss:.4f}")
print(f"  Perplexity:         {perplexity:.2f}")
print(f"  Training time:      {train_elapsed:.1f}s")
print(f"  LoRA adapters:      {model_output_path}")
if save_merged_model:
    print(f"  Merged model:       {merged_output_path}")
print(f"  Metrics JSON:       {metrics_json_path}")
print(f"  Plots:              {plots_dir}")
if executed_notebook_path:
    print(f"  Executed notebook:  {executed_notebook_path}")
print("=" * 60)